In [3]:
import pandas as pd

deliveries = pd.read_csv('/Users/wusiyi/Documents/CASA/dis/DATA/sim_output/deliveries_S1_baseline.csv')  # 用你实际文件名替换
conflicts = pd.read_csv('/Users/wusiyi/Documents/CASA/dis/DATA/sim_output/conflicts_S1_baseline.csv')

print("deliveries columns:", deliveries.columns.tolist())
print("conflicts columns:", conflicts.columns.tolist())

deliveries columns: ['rider_id', 'origin_node', 'elapsed_s', 'time_budget_s', 'overdue', 'distance_m', 'pavement_time_s', 'red_light_violations', 'pavement_conflicts', 'pavement_risk_exposure', 'severe_conflicts']
conflicts columns: ['step', 'sim_time_s', 'rider_id', 'kind', 'severity', 'edge', 'node', 'x', 'y', 'stress_level', 'congestion_index', 'land_use_mix', 'route_trigger']


In [8]:
import sys
sys.path.insert(0, '/Users/wusiyi/Documents/CASA/dis/DATA/files')

from environment import load_environment
from model import SohoDeliveryModel

# data_dir 换成你实际存放 raw/soho_bike_network.graphml 等数据文件的目录
# 大概率就是 /Users/wusiyi/Documents/CASA/dis/DATA
env = load_environment(data_dir='/Users/wusiyi/Documents/CASA/dis/DATA')

model = SohoDeliveryModel(env, scenario="S1_baseline", seed=42)

node_dist = pd.DataFrame([
    {"node": n, "dist_to_boundary": data.get("dist_to_boundary")}
    for n, data in model.env.graph.nodes(data=True)
])
node_dist.to_csv("node_dist.csv", index=False)
print(node_dist.head())
print(node_dist['dist_to_boundary'].isna().sum(), "个节点缺失dist_to_boundary")

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 507，路段 974
  [env] 路段宽度：10 条来自 OSM width 标签，其余用类型默认值
  [env] POI 697 个吸附到 205 个节点
  [env] land_use_mix：均值 0.21，最大 1.00
  [env] 装卸区 10 个吸附到 9 个节点
  [env] 自行车停放 89 个吸附到 62 个节点
         node dist_to_boundary
0      107322             None
1  5239543922             None
2    25417426             None
3   385562683             None
4      107324             None
507 个节点缺失dist_to_boundary


In [9]:
import sys
sys.path.insert(0, '/Users/wusiyi/Documents/CASA/dis/DATA/files')

import pandas as pd
from environment import load_environment
from model import SohoDeliveryModel

# ---------- 1. 加载环境（确保soho_boundary.geojson已放进data_dir） ----------
env = load_environment(data_dir='/Users/wusiyi/Documents/CASA/dis/DATA')

# ---------- 2. 确认边界距离已经写入，没有大面积缺失 ----------
node_dist = pd.DataFrame([
    {"node": n, "dist_to_boundary": data.get("dist_to_boundary")}
    for n, data in env.graph.nodes(data=True)
])
n_missing = node_dist['dist_to_boundary'].isna().sum()
print(f"{n_missing} 个节点缺失 dist_to_boundary（应该是0）")

# 如果还有缺失，先停在这里排查，不要往下跑

# ---------- 3. 建模型，跑180分钟完整S1_baseline ----------
model = SohoDeliveryModel(env, scenario="S1_baseline", seed=42)
n_steps = int(180 * 60 / model.dt)   # 180分钟换算成step数
for _ in range(n_steps):
    model.step()

print(f"跑完，共 {len(model.completed_deliveries)} 单配送，{len(model.conflict_events)} 次冲突事件")

# ---------- 4. 导出三张原始表 ----------
deliveries = pd.DataFrame(model.completed_deliveries)
conflicts = pd.DataFrame(model.conflict_events)

deliveries.to_csv('deliveries.csv', index=False)
conflicts.to_csv('conflicts.csv', index=False)
node_dist.to_csv('node_dist.csv', index=False)

# ---------- 5. deliveries按边界距离分bin ----------
df = deliveries.merge(node_dist, left_on='origin_node', right_on='node', how='left')
df['dist_bin'] = pd.cut(df['dist_to_boundary'], bins=5)

delivery_summary = df.groupby('dist_bin').agg(
    mean_pavement_risk=('pavement_risk_exposure', 'mean'),
    mean_severe_conflicts=('severe_conflicts', 'mean'),
    mean_distance=('distance_m', 'mean'),
    n_deliveries=('rider_id', 'count'),
)
print("=== Deliveries by distance to boundary ===")
print(delivery_summary)

# ---------- 6. conflicts按边界距离分bin ----------
cf = conflicts.merge(node_dist, on='node', how='left')
cf['dist_bin'] = pd.cut(cf['dist_to_boundary'], bins=5)

conflict_summary = cf.groupby('dist_bin').agg(
    n_conflicts=('rider_id', 'count'),
    mean_congestion=('congestion_index', 'mean'),
    mean_stress=('stress_level', 'mean'),
)
print("=== Conflicts by distance to boundary ===")
print(conflict_summary)

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 507，路段 974
  [env] 路段宽度：10 条来自 OSM width 标签，其余用类型默认值
  [env] POI 697 个吸附到 205 个节点
  [env] land_use_mix：均值 0.21，最大 1.00
  [env] 装卸区 10 个吸附到 9 个节点
  [env] 自行车停放 89 个吸附到 62 个节点
507 个节点缺失 dist_to_boundary（应该是0）
跑完，共 1810 单配送，28995 次冲突事件


ValueError: Bin edges must be unique: Index([nan, nan, nan, nan, nan, nan], dtype='float64').
You can drop duplicate edges by setting the 'duplicates' kwarg

In [10]:
print(deliveries['origin_node'].dtype, node_dist['node'].dtype)
print(deliveries['origin_node'].head())
print(node_dist['node'].head())

int64 int64
0       9512925
1      25501320
2        107793
3    9167793401
4       9791152
Name: origin_node, dtype: int64
0        107322
1    5239543922
2      25417426
3     385562683
4        107324
Name: node, dtype: int64


In [11]:
deliveries['origin_node'] = deliveries['origin_node'].astype(str)
node_dist['node'] = node_dist['node'].astype(str)
df = deliveries.merge(node_dist, left_on='origin_node', right_on='node', how='left')
print(df['dist_to_boundary'].isna().sum(), "行没匹配上")

1810 行没匹配上


In [12]:
import os
print(os.path.exists('/Users/wusiyi/Documents/CASA/dis/DATA/soho_boundary.geojson'))

True


In [13]:
import importlib
import environment
importlib.reload(environment)

from environment import load_environment
from model import SohoDeliveryModel
import pandas as pd

env = load_environment(data_dir='/Users/wusiyi/Documents/CASA/dis/DATA')

  [env] 路网已投影：EPSG:32630
  [env] 路网节点 507，路段 974
  [env] 路段宽度：10 条来自 OSM width 标签，其余用类型默认值
  [env] POI 697 个吸附到 205 个节点
  [env] land_use_mix：均值 0.21，最大 1.00
  [env] 装卸区 10 个吸附到 9 个节点
  [env] 自行车停放 89 个吸附到 62 个节点


In [1]:
import sys
sys.path.insert(0, '/Users/wusiyi/Documents/CASA/dis/DATA/files')

import pandas as pd
from environment import load_environment
from model import SohoDeliveryModel

# ---------- 1. 加载环境 ----------
env = load_environment(data_dir='/Users/wusiyi/Documents/CASA/dis/DATA')

# ---------- 2. 确认边界距离已经写入 ----------
node_dist = pd.DataFrame([
    {"node": n, "dist_to_boundary": data.get("dist_to_boundary")}
    for n, data in env.graph.nodes(data=True)
])
n_missing = node_dist['dist_to_boundary'].isna().sum()
print(f"{n_missing} 个节点缺失 dist_to_boundary（应该是0）")

/Users/wusiyi/Documents/CASA/dis/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  [env] 路网已投影：EPSG:32630
  [env] 路网节点 507，路段 974
  [env] 路段宽度：10 条来自 OSM width 标签，其余用类型默认值
  [env] POI 697 个吸附到 205 个节点
  [env] land_use_mix：均值 0.21，最大 1.00
  [env] 装卸区 10 个吸附到 9 个节点
  [env] 自行车停放 89 个吸附到 62 个节点
  [env] 边界已加载并投影到 EPSG:32630
0 个节点缺失 dist_to_boundary（应该是0）


In [2]:
# ---------- 3. 建模型，跑180分钟完整S1_baseline ----------
model = SohoDeliveryModel(env, scenario="S1_baseline", seed=42)
n_steps = int(180 * 60 / model.dt)
for _ in range(n_steps):
    model.step()

print(f"跑完，共 {len(model.completed_deliveries)} 单配送，{len(model.conflict_events)} 次冲突事件")

# ---------- 4. 导出三张原始表 ----------
deliveries = pd.DataFrame(model.completed_deliveries)
conflicts = pd.DataFrame(model.conflict_events)

deliveries.to_csv('deliveries.csv', index=False)
conflicts.to_csv('conflicts.csv', index=False)
node_dist.to_csv('node_dist.csv', index=False)

# ---------- 5. deliveries按边界距离分bin ----------
df = deliveries.merge(node_dist, left_on='origin_node', right_on='node', how='left')
df['dist_bin'] = pd.cut(df['dist_to_boundary'], bins=5)

delivery_summary = df.groupby('dist_bin').agg(
    mean_pavement_risk=('pavement_risk_exposure', 'mean'),
    mean_severe_conflicts=('severe_conflicts', 'mean'),
    mean_distance=('distance_m', 'mean'),
    n_deliveries=('rider_id', 'count'),
)
print("=== Deliveries by distance to boundary ===")
print(delivery_summary)

# ---------- 6. conflicts按边界距离分bin ----------
cf = conflicts.merge(node_dist, on='node', how='left')
cf['dist_bin'] = pd.cut(cf['dist_to_boundary'], bins=5)

conflict_summary = cf.groupby('dist_bin').agg(
    n_conflicts=('rider_id', 'count'),
    mean_congestion=('congestion_index', 'mean'),
    mean_stress=('stress_level', 'mean'),
)
print("=== Conflicts by distance to boundary ===")
print(conflict_summary)

跑完，共 1810 单配送，28995 次冲突事件
=== Deliveries by distance to boundary ===
                    mean_pavement_risk  mean_severe_conflicts  mean_distance  \
dist_bin                                                                       
(-0.125, 44.502]             21.389798               1.641141     781.259580   
(44.502, 88.907]             17.935277               1.216842     756.222936   
(88.907, 133.312]            15.695134               1.442105     775.619915   
(133.312, 177.717]           13.167002               1.386598     694.494673   
(177.717, 222.122]           17.178053               2.168421     699.734978   

                    n_deliveries  
dist_bin                          
(-0.125, 44.502]             666  
(44.502, 88.907]             475  
(88.907, 133.312]            380  
(133.312, 177.717]           194  
(177.717, 222.122]            95  
=== Conflicts by distance to boundary ===
                    n_conflicts  mean_congestion  mean_stress
dist_bin             

In [3]:
import json

for name in ["soho_loading_bays.geojson", "soho_cycle_parking.geojson"]:
    path = f"/Users/wusiyi/Documents/CASA/dis/DATA/raw/{name}"
    with open(path) as f:
        data = json.load(f)
    props = data["features"][0]["properties"]
    print(name, "->", list(props.keys()))
    print(props)
    print()


soho_loading_bays.geojson -> ['street', 'lat', 'lon']
{'street': 'Frith Street', 'lat': 51.514769, 'lon': -0.132311}

soho_cycle_parking.geojson -> ['FEATURE_ID', 'SVDATE', 'PRK_CARR', 'PRK_COVER', 'PRK_SECURE', 'PRK_LOCKER', 'PRK_SHEFF', 'PRK_MSTAND', 'PRK_PSTAND', 'PRK_HOOP', 'PRK_POST', 'PRK_BUTERF', 'PRK_WHEEL', 'PRK_HANGAR', 'PRK_TIER', 'PRK_OTHER', 'PRK_PROVIS', 'PRK_CPT', 'BOROUGH', 'PHOTO1_URL', 'PHOTO2_URL']
{'FEATURE_ID': 'RWG057654', 'SVDATE': '2017-06-28T00:00:00', 'PRK_CARR': 'FALSE', 'PRK_COVER': 'FALSE', 'PRK_SECURE': 'FALSE', 'PRK_LOCKER': 'FALSE', 'PRK_SHEFF': 'FALSE', 'PRK_MSTAND': 'FALSE', 'PRK_PSTAND': 'FALSE', 'PRK_HOOP': 'TRUE', 'PRK_POST': 'FALSE', 'PRK_BUTERF': 'FALSE', 'PRK_WHEEL': 'FALSE', 'PRK_HANGAR': 'FALSE', 'PRK_TIER': 'FALSE', 'PRK_OTHER': 'FALSE', 'PRK_PROVIS': 1.0, 'PRK_CPT': 2.0, 'BOROUGH': 'Westminster', 'PHOTO1_URL': 'https://cycleassetimages.data.tfl.gov.uk/RWG057654_1.jpg', 'PHOTO2_URL': 'https://cycleassetimages.data.tfl.gov.uk/RWG057654_2.jpg'}
